# 05 - Validation against openrouteservice
Recompute the network metrics of notebook 04 for 10 random buildings using the openrouteservice (HeiGIT) API
(`foot-walking` matrix and isochrones) and compare them with the values in `output/buildings_network.gpkg`.
Nothing in the original process is changed; this notebook only reports the differences.

In [1]:
import os
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import requests
import networkx as nx
import osmnx as ox
import folium

In [2]:
API_KEY = os.environ["OPEN_ROUTE_SERVICE"]

## Load data and sample 10 buildings
Same layers and filters as notebook 04. `fid` is the GeoPackage feature id (the `fid` field in QGIS).

In [3]:
CRS = 32721
WALK = 1200
gp = "data/geopackage"
ORS = "https://api.heigit.org/openrouteservice/v2"
HEAD = {"Authorization": API_KEY}

bld = gpd.read_file("output/buildings_network.gpkg", fid_as_index=True).to_crs(CRS)
sample = bld.sample(10, random_state=42)
sample.index.name = "fid"

read = lambda f: gpd.read_file(f"{gp}/{f}.gpkg").to_crs(CRS).reset_index(drop=True)
amenities, coast, green, transport = read("amenities"), read("coast_beaches"), read("green_areas"), read("public_transport")
coastline = coast[coast["natural"] == "coastline"]
beaches = coast[coast["natural"] == "beach"]
is_park = (green["leisure"].isin(["park", "garden", "playground"]) | (green["place"] == "square")).values
parks = green[is_park]
hubs = transport[(transport["amenity"] == "bus_station") | (transport["public_transport"] == "station")]
stops = transport[transport["highway"] == "bus_stop"]

sample[["s2_id", "geohash", "area_in_meters", "livability"]]

,s2_id,geohash,area_in_meters,livability
fid,,,,
5717,-7782220156096217088,6cbcq9ng,96.5040,36.0
34186,-7782220156096217088,6cbcmy0x,63.8458,45.1
42918,-7782220156096217088,6cbcmm9h,140.5635,71.7
31480,-7782220156096217088,6cbcmzjc,32.6933,34.0
75274,-7782220156096217088,6cbcm6vw,55.4486,72.0
60785,-7782220156096217088,6cbcsbff,67.5029,35.4
60580,-7782220156096217088,6cbcsb42,43.4924,25.2
15046,-7782220156096217088,6cbcmd6n,16.4364,46.5
43444,-7782220156096217088,6cbcmm53,59.7674,74.5


## ORS helpers
ORS routes between points, so every feature is reduced to its point closest (straight line) to the building
(the building itself if it lies inside a polygon). Beaches and coastline are long features, so their outlines are
densified every 100 m and each vertex is a candidate. Only candidates within 1200 m straight line, plus the 49 nearest,
are sent to the matrix (in chunks of 49, respecting the 40 requests/min limit).

In [4]:
def matrix(p, targets):
    ll = gpd.GeoSeries([p, *targets], crs=CRS).to_crs(4326)
    locs = np.c_[ll.x, ll.y].tolist()
    out = []
    for i in range(1, len(locs), 49):
        chunk = locs[i:i + 49]
        r = requests.post(f"{ORS}/matrix/foot-walking", headers=HEAD, json={
            "locations": [locs[0]] + chunk, "sources": [0],
            "destinations": list(range(1, len(chunk) + 1)), "metrics": ["distance"]})
        r.raise_for_status()
        j = r.json()
        out += j["distances"][0]
        time.sleep(1.6)
    return np.array(out, dtype=float), j["sources"][0].get("snapped_distance", 0.0)

def ors_layer(p, geoms):
    t = gpd.GeoSeries(shapely.get_point(shapely.shortest_line(geoms.values, p), 0), crs=CRS)
    straight = t.distance(p)
    keep = ((straight <= WALK) | (straight.rank(method="first") <= 49)).values
    d = pd.Series(np.inf, index=t.index)
    d[keep], snap = matrix(p, t[keep])
    return d, straight, snap

def outline_points(layer):
    g = layer.geometry
    g = g.where(g.geom_type.str.contains("Line"), g.boundary).segmentize(100)
    return gpd.GeoSeries(gpd.points_from_xy(*g.get_coordinates().values.T), crs=CRS)

beach_pts, coast_pts = outline_points(beaches), outline_points(coastline)

## Compute ORS metrics
Same definitions as notebook 04. The isochrone (1200 m walking distance) gives a second, independent count of features within 15 minutes.

In [5]:
HEAD = {"Authorization": API_KEY}  # used by matrix() and the isochrone request below

ors, iso, snaps, targets = {}, {}, {}, {}
for fid, b in sample.iterrows():
    p = b.geometry.centroid
    am, am_s, snaps[fid] = ors_layer(p, amenities.geometry)
    gr, _, _ = ors_layer(p, green.geometry)
    st, _, _ = ors_layer(p, stops.geometry)
    hb, _, _ = ors_layer(p, hubs.geometry)
    be, _, _ = ors_layer(p, beach_pts)
    co, _, _ = ors_layer(p, coast_pts)
    ors[fid] = {
        "nd_amenity5_m": am.nsmallest(5).mean(), "nd_green5_m": gr.nsmallest(5).mean(),
        "nd_hub_m": hb.min(), "nd_bus_stop_m": st.min(), "nd_beach_m": be.min(), "nd_coast_m": co.min(),
        "detour_amenity": max(am.min() / max(am_s[am.idxmin()], 50), 1),
        "parks_15min": (gr[is_park] <= WALK).sum(), "amenities_15min": (am <= WALK).sum(),
        "bus_stops_15min": (st <= WALK).sum()}
    # ORS-nearest destination point of each single-target variable (used by the visual checks)
    targets[fid] = {k: shapely.get_point(shapely.shortest_line(g.iloc[d.argmin()], p), 0) for k, g, d in
                    [("nd_hub_m", hubs.geometry, hb), ("nd_bus_stop_m", stops.geometry, st),
                     ("nd_beach_m", beach_pts, be), ("nd_coast_m", coast_pts, co)]}

    ll = gpd.GeoSeries([p], crs=CRS).to_crs(4326)
    r = requests.post(f"{ORS}/isochrones/foot-walking", headers=HEAD, json={
        "locations": [[ll.x[0], ll.y[0]]], "range": [WALK], "range_type": "distance"})
    r.raise_for_status()
    poly = gpd.GeoDataFrame.from_features(r.json(), crs=4326).to_crs(CRS).union_all()
    iso[fid] = {"parks_15min": parks.intersects(poly).sum(), "amenities_15min": amenities.intersects(poly).sum(),
                "bus_stops_15min": stops.intersects(poly).sum()}
    time.sleep(3)

ors, iso = pd.DataFrame(ors).T, pd.DataFrame(iso).T
print("ORS snapped distance (m) - building centroid to ORS road network:", pd.Series(snaps).round(1).to_dict())

ORS snapped distance (m) - building centroid to ORS road network: {5717: 43.1, 34186: 29.7, 42918: 13.9, 31480: 16.4, 75274: 23.0, 60785: 14.4, 60580: 28.8, 15046: 30.7, 43444: 16.1, 663: 26.0}


In [6]:
import requests

body = {"locations":[[9.70093,48.477473],[9.207916,49.153868],[37.573242,55.801281],[115.663757,38.106467]]}

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': API_KEY,
    'Content-Type': 'application/json; charset=utf-8'
}
call = requests.post('https://api.heigit.org/openrouteservice/v2/matrix/foot-walking', json=body, headers=headers)

print(call.status_code, call.reason)
print(call.text)

200 OK
{"destinations":[{"location":[9.700817,48.476406],"snapped_distance":118.9},{"location":[9.207828,49.153877],"snapped_distance":6.49},{"location":[37.572963,55.801279],"snapped_distance":17.44},{"location":[115.665017,38.100717],"snapped_distance":648.79}],"durations":[[0.0,73687.48,1675000.38,7345629.0],[73687.48,0.0,1605707.38,7276336.0],[1675000.38,1605707.38,0.0,5692378.5],[7345629.0,7276336.0,5692378.5,0.0]],"sources":[{"location":[9.700817,48.476406],"snapped_distance":118.9},{"location":[9.207828,49.153877],"snapped_distance":6.49},{"location":[37.572963,55.801279],"snapped_distance":17.44},{"location":[115.665017,38.100717],"snapped_distance":648.79}],"metadata":{"query":{"locations":[[9.70093,48.477473],[9.207916,49.153868],[37.573242,55.801281],[115.663757,38.106467]],"profile":"foot-walking","profileName":"foot-walking","responseType":"json"},"attribution":"openrouteservice.org | OpenStreetMap contributors","engine":{"build_date":"2026-11-18T10:33:03Z","graph_date":"2

## Comparison per building
`notebook` = value in `output/buildings_network.gpkg` (includes the building-to-street snap leg);
`ors` = ORS matrix (excludes the snap leg); `ors_iso` = count inside the ORS isochrone.

In [7]:
cols = list(ors.columns)
cmp = pd.concat({"notebook": sample[cols].stack(), "ors": ors.stack(), "ors_iso": iso.stack()}, axis=1).astype(float)
cmp.index.names = ["fid", "variable"]
cmp["diff"] = cmp["ors"] - cmp["notebook"]
cmp["diff_%"] = 100 * cmp["diff"] / cmp["notebook"].replace(0, np.nan)
pd.set_option("display.max_rows", 100)
cmp.round(2)

notebook      ors  ors_iso     diff  diff_%
fid   variable                                                    
5717  nd_amenity5_m     3025.96  3172.91      NaN   146.95    4.86
      nd_green5_m        324.42   365.31      NaN    40.89   12.60
      nd_hub_m          4837.92  5836.37      NaN   998.45   20.64
      nd_bus_stop_m      262.95   289.97      NaN    27.02   10.28
      nd_beach_m         361.11   318.09      NaN   -43.02  -11.91
      nd_coast_m        2273.06   335.91      NaN -1937.15  -85.22
      detour_amenity       1.19     1.23      NaN     0.04    3.68
      parks_15min          0.00     0.00      0.0     0.00     NaN
      amenities_15min      0.00     0.00      0.0     0.00     NaN
      bus_stops_15min      2.00     2.00      2.0     0.00    0.00
34186 nd_amenity5_m      769.80   882.37      NaN   112.57   14.62
      nd_green5_m        577.41   597.35      NaN    19.94    3.45
      nd_hub_m          1764.21  1967.10      NaN   202.89   11.50
      nd_bus_stop_m      196.37   213.07      NaN    16.70    8.50
      nd_beach_m        4038.19  4180.68      NaN   142.49    3.53
      nd_coast_m        4118.11  4195.52      NaN    77.41    1.88
      detour_amenity       1.27     1.25      NaN    -0.02   -1.51
      parks_15min          1.00     3.00      3.0     2.00  200.00
      amenities_15min      9.00     7.00     10.0    -2.00  -22.22
      bus_stops_15min     23.00    22.00     23.0    -1.00   -4.35
42918 nd_amenity5_m      182.22   167.47      NaN   -14.75   -8.09
      nd_green5_m        117.87   144.78      NaN    26.91   22.83
      nd_hub_m          1486.65  1505.44      NaN    18.79    1.26
      nd_bus_stop_m      310.40   368.86      NaN    58.46   18.83
      nd_beach_m        1962.43  2054.68      NaN    92.25    4.70
      nd_coast_m        2042.34  2054.68      NaN    12.34    0.60
      detour_amenity       1.65     1.59      NaN    -0.06   -3.92
      parks_15min          9.00     8.00     11.0    -1.00  -11.11
      amenities_15min     24.00    26.00     28.0     2.00    8.33
      bus_stops_15min     26.00    24.00     30.0    -2.00   -7.69
31480 nd_amenity5_m     1578.41  1714.59      NaN   136.18    8.63
      nd_green5_m        529.35   535.98      NaN     6.63    1.25
      nd_hub_m          1997.55  2725.38      NaN   727.83   36.44
      nd_bus_stop_m      446.22   443.54      NaN    -2.68   -0.60
      nd_beach_m        5115.60  5147.14      NaN    31.54    0.62
      nd_coast_m        5195.52  5161.98      NaN   -33.54   -0.65
      detour_amenity       1.28     1.51      NaN     0.23   18.33
      parks_15min          3.00     3.00      3.0     0.00    0.00
      amenities_15min      1.00     0.00      1.0    -1.00 -100.00
      bus_stops_15min      5.00     1.00      6.0    -4.00  -80.00
75274 nd_amenity5_m      592.43   653.39      NaN    60.96   10.29
      nd_green5_m        333.59   322.25      NaN   -11.34   -3.40
      nd_hub_m           452.18   451.54      NaN    -0.64   -0.14
      nd_bus_stop_m      169.22   188.83      NaN    19.61   11.59
      nd_beach_m        1121.63  1143.98      NaN    22.35    1.99
      nd_coast_m        1191.03  1146.76      NaN   -44.27   -3.72
      detour_amenity       1.15     1.09      NaN    -0.06   -5.49
      parks_15min          9.00     8.00      9.0    -1.00  -11.11
      amenities_15min      8.00     6.00      8.0    -2.00  -25.00
      bus_stops_15min     24.00    24.00     30.0     0.00    0.00
60785 nd_amenity5_m     2126.26  2346.82      NaN   220.56   10.37
      nd_green5_m        190.06   219.93      NaN    29.87   15.72
      nd_hub_m          3330.64  3516.67      NaN   186.03    5.59
      nd_bus_stop_m      468.99   481.02      NaN    12.03    2.57
      nd_beach_m        3405.15  3582.82      NaN   177.67    5.22
      nd_coast_m        3724.79  3582.82      NaN  -141.97   -3.81
      detour_amenity       1.42     1.15      NaN    -0.27  -19.12
      parks_15min          1.00     1.00      1.0     0.00    0.00
   

## Summary per variable

In [8]:
g = cmp.groupby("variable", sort=False)
pd.DataFrame({"mean_notebook": g["notebook"].mean(), "mean_ors": g["ors"].mean(),
              "mean_abs_diff": g["diff"].apply(lambda d: d.abs().mean()),
              "mean_abs_diff_%": g["diff_%"].apply(lambda d: d.abs().mean()),
              "ors_higher (of 10)": g["diff"].apply(lambda d: (d > 0).sum()),
              "mean_abs_diff_iso": (cmp["ors_iso"] - cmp["notebook"]).abs().groupby("variable", sort=False).mean()}).round(2)

,mean_notebook,mean_ors,mean_abs_diff,mean_abs_diff_%,ors_higher (of 10),mean_abs_diff_iso
variable,,,,,,
nd_amenity5_m,1534.48,1639.15,144.45,11.88,7,NaN
nd_green5_m,387.44,388.14,24.14,8.16,5,NaN
nd_hub_m,2348.08,2677.47,368.03,13.13,8,NaN
nd_bus_stop_m,394.49,380.50,40.75,11.55,5,NaN
nd_beach_m,2285.40,2323.83,99.60,6.21,7,NaN
nd_coast_m,2640.72,2331.48,329.04,17.39,3,NaN
detour_amenity,1.48,1.29,0.31,17.41,3,NaN
parks_15min,3.80,3.80,0.40,27.78,1,0.8
amenities_15min,8.00,8.00,1.00,50.79,3,0.8


## Known sources of difference
- **Snap leg**: notebook 04 adds the straight-line distance from the building centroid to its nearest street node; ORS starts at the
  snapped point on the nearest road segment and does not add that leg (see the snapped distances printed above).
- **Network data**: ORS uses its own OSM extract and `foot-walking` profile (which may exclude some ways); notebook 04 uses the
  OSMnx `all` network downloaded in notebook 02, largest connected component only.
- **Polygon and line targets**: notebook 04 links every street node within 100 m of a feature; here each feature is reached at its
  nearest point (beaches/coast: densified outline vertices), so polygon/line distances are approximations.
- **Isochrone counts** depend on ORS's polygon generalisation, so they can differ from the matrix counts near the 1200 m limit.

## Visual checks
Web maps for the cases that stand out. The notebook 04 route is rebuilt on the same street graph and with the same feature
linking (nodes within 100 m + nearest node, plus the building snap leg). The printed "rebuilt" distance should match the stored value.
The ORS route comes from the `foot-walking` directions service to the ORS-nearest target.

In [9]:
nodes = gpd.read_file(f"{gp}/street_network.gpkg", layer="nodes").set_index("osmid")
edges = gpd.read_file(f"{gp}/street_network.gpkg", layer="edges").set_index(["u", "v", "key"])
G = ox.truncate.largest_component(ox.project_graph(ox.graph_from_gdfs(nodes, edges), to_crs=CRS))
U = ox.convert.to_undirected(G)  # walking ignores one-way restrictions
gnodes = ox.graph_to_gdfs(G, edges=False)[["geometry"]]
layers = {"nd_hub_m": hubs, "nd_bus_stop_m": stops, "nd_beach_m": beaches, "nd_coast_m": coastline}

def osmnx_route(p, layer):
    # same feature linking as notebook 04: nearest node of each feature + every node within 100 m
    (_, n), w = gnodes.sindex.nearest(layer.geometry, return_distance=True, return_all=False)
    nn, ff = layer.sindex.query(gnodes.geometry, predicate="dwithin", distance=100)
    acc = pd.Series(np.r_[w, gnodes.geometry.values[nn].distance(layer.geometry.values[ff])],
                    index=gnodes.index[np.r_[n, nn]]).groupby(level=0).min()
    (_, (bn,)), (bs,) = gnodes.sindex.nearest(p, return_distance=True, return_all=False)
    start = gnodes.index[bn]
    dist, path = nx.single_source_dijkstra(U, start, weight="length")
    total = (pd.Series(dist).reindex(acc.index) + acc + bs).dropna()
    end = total.idxmin()
    segs = [min(U[u][v].values(), key=lambda e: e["length"]).get(
                "geometry", shapely.LineString([gnodes.geometry[u], gnodes.geometry[v]]))
            for u, v in zip(path[end][:-1], path[end][1:])]
    q = gnodes.geometry[end]
    dest = shapely.get_point(shapely.shortest_line(layer.geometry.values[layer.distance(q).argmin()], q), 0)
    legs = shapely.MultiLineString([[p, gnodes.geometry[start]], [q, dest]])
    return total.min(), shapely.MultiLineString(segs), legs, dest

def ors_route(p, q):
    ll = gpd.GeoSeries([p, q], crs=CRS).to_crs(4326)
    r = requests.post(f"{ORS}/directions/foot-walking/geojson", headers=HEAD, json={"coordinates": np.c_[ll.x, ll.y].tolist()})
    r.raise_for_status()
    f = r.json()["features"][0]
    return shapely.geometry.shape(f["geometry"]), f["properties"]["summary"].get("distance", 0.0)

# significant = single-destination distance with |diff| > 200 m and |diff| > 20 %
sel = cmp.loc[(slice(None), list(layers)), :]
sel = sel[(sel["diff"].abs() > 200) & (sel["diff_%"].abs() > 20)]
print(len(sel), "significant cases")

to_ll = lambda *g: gpd.GeoSeries(list(g), crs=CRS).to_crs(4326)
for (fid, var), row in sel.iterrows():
    p = sample.geometry[fid].centroid
    d_nx, r_nx, legs, dest_nx = osmnx_route(p, layers[var])
    r_ors, d_ors = ors_route(p, targets[fid][var])
    print(f"fid {fid} {var}: stored {row['notebook']:.0f} m | OSMnx rebuilt {d_nx:.0f} m | "
          f"ORS matrix {row['ors']:.0f} m | ORS directions {d_ors:.0f} m")

    m = folium.Map(tiles="OpenStreetMap")
    folium.GeoJson(sample.loc[[fid]].to_crs(4326).geometry, tooltip=f"building {fid}",
                   style_function=lambda _: {"color": "black", "fillColor": "black", "fillOpacity": 0.6}).add_to(m)
    folium.GeoJson(to_ll(r_nx), tooltip=f"OSMnx route ({d_nx:.0f} m)",
                   style_function=lambda _: {"color": "blue", "weight": 6, "opacity": 0.6}).add_to(m)
    folium.GeoJson(to_ll(legs), tooltip="OSMnx snap / access legs",
                   style_function=lambda _: {"color": "blue", "weight": 2, "dashArray": "6"}).add_to(m)
    folium.GeoJson(r_ors, tooltip=f"ORS route ({d_ors:.0f} m)",
                   style_function=lambda _: {"color": "red", "weight": 3}).add_to(m)
    for g, c, t in [(dest_nx, "blue", "OSMnx destination"), (targets[fid][var], "red", "ORS destination")]:
        pt = to_ll(g)[0]
        folium.Marker([pt.y, pt.x], tooltip=t, icon=folium.Icon(color=c)).add_to(m)
    b = gpd.GeoSeries([*to_ll(r_nx, legs, targets[fid][var]), r_ors]).total_bounds
    m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
    folium.map.Marker([b[3], b[0]], icon=folium.DivIcon(html=f"<b style='font-size:14px;white-space:nowrap'>fid {fid} - {var}</b>")).add_to(m)
    display(m)

5 significant cases


fid 5717 nd_hub_m: stored 4838 m | OSMnx rebuilt 4838 m | ORS matrix 5836 m | ORS directions 5836 m


fid 5717 nd_coast_m: stored 2273 m | OSMnx rebuilt 2273 m | ORS matrix 336 m | ORS directions 336 m


fid 31480 nd_hub_m: stored 1998 m | OSMnx rebuilt 1998 m | ORS matrix 2725 m | ORS directions 2725 m


fid 663 nd_hub_m: stored 4154 m | OSMnx rebuilt 4154 m | ORS matrix 5206 m | ORS directions 5206 m


fid 663 nd_coast_m: stored 1058 m | OSMnx rebuilt 1058 m | ORS matrix 486 m | ORS directions 486 m


## Considerations on the significant cases
Blue = OSMnx (notebook 04, rebuilt; it matches the stored value to within 0.1 m), red = ORS `foot-walking`.

- **fid 5717 - `nd_coast_m` (notebook 2273 m vs ORS 336 m).** Neither value is right, but notebook 04 is the further off.
  The nearest coastline point is ~445 m away in a straight line. The closest OSMnx street node is ~230 m from it, so no node is
  within the 100 m linking distance there. The coastline is a long line feature with only one "nearest node" link, so notebook 04
  has to walk to a distant stretch where a street comes within 100 m of the shore. ORS snaps the coast vertex to its nearest road
  and does not count the remaining ~230 m to the shore, so 336 m is too low. A realistic value is ~500-550 m: the 270 m OSMnx path
  to the street closest to that point, plus the ~230 m access gap. **This is a weakness of the 100 m linking for coastline/beach
  wherever the shore has no street close to it (dunes, rocky coast).**
- **fid 5717 - `nd_hub_m` (4838 m vs 5836 m).** Same destination (the Terminal de Ómnibus polygon). About 4.1 km of the OSMnx
  route runs along Rambla Lorenzo Batlle Pacheco (secondary road, the coastal road), and ORS avoids it for a longer inland path.
  Notebook 04 also saves ~170 m because the terminal polygon links to any street node within 100 m. The OSMnx `all` network
  ignores pedestrian suitability, so notebook 04 likely gives a lower bound here.
- **fid 31480 - `nd_hub_m` (1998 m vs 2725 m).** Same destination (CODESA polygon). The OSMnx route uses residential/tertiary
  streets (Agapito Paraberá, Juan Alberto Caracará, Meboipe, ~1.5 km) that ORS does not follow. These ways are probably missing
  from ORS's OSM extract, or tagged so that the foot profile avoids them. Another ~260 m comes from linking the polygon through
  nodes within 100 m, while ORS goes to a single point on its edge.
- **fid 663 - `nd_hub_m` (4154 m vs 5206 m).** Same destination (Terminal de Ómnibus Maldonado). About 2.3 km of the OSMnx
  route runs along Avenida a la Laguna (tertiary), and ORS avoids it for a ~1 km longer path. This is the same pattern as
  fid 5717: suburban arterials that the OSMnx `all` network uses freely, but that ORS's foot profile skips or does not have.
- **fid 663 - `nd_coast_m` (1058 m vs 486 m).** Same issue as fid 5717 coast, on a smaller scale. The nearest shore is ~555 m away
  in a straight line, and the closest street node is ~130 m from it, beyond the 100 m link. Notebook 04 reaches the coast
  ~790 m away instead. ORS stops ~125 m short of the shore and does not count that gap. A realistic value is ~600 m (the OSMnx
  path to the nearest street plus the gap), so notebook 04 overestimates by ~450 m.

**Overall.** The large differences do not come from errors in the routing itself. They come from (1) the 100 m feature linking,
which helps notebook 04 for polygons and penalises it heavily for shores without nearby streets, and (2) ORS's pedestrian profile
and network differing from the OSMnx `all` network on long routes. Hub distances are only a small part of the index. Two of the ten buildings show the coast
artefact, so it matters more: it affects every building near a stretch of coast that has no street.
Below the threshold, fid 15046 `nd_bus_stop_m` (367 m vs 181 m, -51%) is the only other notable gap.